# Trading Intelligent — CH02 — Notebook 2
## ARIMA : identification, diagnostics et prévision hors échantillon

**Auteur académique : Tawfik Masrour**  
**Version : 2.2**  
**Dataset : `TI_CH02_TS_MODELS_V1`**

## Question directrice
Un ARIMA apporte-t-il réellement quelque chose lorsque la comparaison est faite honnêtement face à plusieurs baselines raisonnables et sur un futur resté intact ?

## Ce que vous allez apprendre
1. séparer proprement entraînement et futur ;
2. distinguer prévision multi-pas à origine fixe et protocole walk-forward ;
3. construire deux baselines : persistance et drift ;
4. comparer des modèles candidats par AIC/BIC **sur l'entraînement** ;
5. diagnostiquer les résidus avec ACF et Ljung–Box en tenant compte de `model_df` ;
6. évaluer ARIMA et baselines sur exactement la même période future.

## Prérequis
NB01, différenciation, AR/MA, ACF/PACF, ADF/KPSS, MAE et RMSE.

## Comment utiliser ce notebook
Le jeu de test doit rester fermé jusqu'à la section d'évaluation. Ne changez pas de modèle après avoir vu sa performance test puis ne réutilisez pas ce même test comme preuve finale.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

SEED = 20260930
DATASET_ID = "TI_CH02_TS_MODELS_V1"
DATASET_FILENAME = "TI_CH02_TS_Models_v1.csv"

# Palette canonique Trading Intelligent
ACADEMIC_BLUE = "#173B57"
PETROL = "#2C7A7B"
WARM_GOLD = "#C79A3B"
MUTED_RED = "#A94B4B"
NEUTRAL_GRAY = "#66717D"

plt.rcParams.update({
    "figure.figsize": (10, 4.5),
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "axes.edgecolor": NEUTRAL_GRAY,
    "xtick.color": NEUTRAL_GRAY,
    "ytick.color": NEUTRAL_GRAY,
})


def generate_dataset(seed=SEED, n=520):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2024-01-02", periods=n)
    eps = rng.normal(0, 1, n)

    def ar1(phi):
        x = np.zeros(n)
        for t in range(1, n):
            x[t] = phi * x[t-1] + eps[t]
        return x

    def ma1(theta):
        x = eps.copy()
        x[1:] += theta * eps[:-1]
        return x

    random_walk = 100 + np.cumsum(0.12 + rng.normal(0, 1.1, n))
    innov = rng.normal(0, 0.8, n)
    inc = np.zeros(n)
    for t in range(1, n):
        inc[t] = 0.35 * inc[t-1] + innov[t] + 0.45 * innov[t-1] + 0.04
    arima_price = 100 + np.cumsum(inc)

    return pd.DataFrame({
        "date": dates,
        "white_noise": eps,
        "ar1_phi_075": ar1(0.75),
        "ar1_phi_neg_065": ar1(-0.65),
        "ma1_theta_070": ma1(0.70),
        "random_walk_price": random_walk,
        "arima_price": arima_price,
    })


def locate_dataset(filename=DATASET_FILENAME):
    here = Path.cwd()
    roots = [here, here.parent, here.parent.parent]
    for root in roots:
        for sub in [Path('.'), Path('datasets'), Path('data'), Path('resources')]:
            p = (root / sub / filename).resolve()
            if p.exists():
                return p
    return None


path = locate_dataset()
if path is not None:
    df = pd.read_csv(path, parse_dates=['date'])
    print(f"{DATASET_ID} chargé : {len(df)} lignes, {len(df.columns)} colonnes")
else:
    df = generate_dataset()
    print(f"{DATASET_ID} absent localement : version déterministe régénérée en mémoire.")

assert len(df) == 520
assert not df.isna().any().any()
df.head()


In [ ]:
def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred))**2)))

def fmt_p(p):
    if p < 1e-10:
        return f"p = {p:.2e} (< 1e-10)"
    if p < 1e-4:
        return f"p = {p:.2e}"
    return f"p = {p:.4f}"


## 1. Série de travail et fréquence explicite

`arima_price` est un niveau intégré obtenu en cumulant une dynamique ARMA. Nous fixons explicitement la fréquence **business-day (`B`)** afin que Statsmodels ne doive pas l'inférer et que le `Run all` reste propre.


In [ ]:
y = pd.Series(
    df.arima_price.to_numpy(),
    index=pd.DatetimeIndex(df.date),
    name="price"
).asfreq("B")
assert not y.isna().any()

fig, axs = plt.subplots(2, 1, figsize=(11, 7), sharex=False)
axs[0].plot(y, color=ACADEMIC_BLUE); axs[0].set_title("Niveau"); axs[0].grid(alpha=0.2)
axs[1].plot(y.diff(), color=PETROL); axs[1].set_title("Première différence"); axs[1].grid(alpha=0.2)
plt.tight_layout(); plt.show()

adf_level = adfuller(y, autolag="AIC")
adf_diff = adfuller(y.diff().dropna(), autolag="AIC")
print("ADF niveau     :", fmt_p(adf_level[1]))
print("ADF différence :", fmt_p(adf_diff[1]))


## 2. Frontière d'information : train / test

Nous réservons les 60 dernières observations. Elles représentent un futur qui ne doit pas intervenir dans l'estimation ni dans la sélection AIC/BIC.


In [ ]:
h = 60
train = y.iloc[:-h]
test = y.iloc[-h:]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train.index, train, label="Entraînement", color=ACADEMIC_BLUE)
ax.plot(test.index, test, label="Test futur", color=PETROL)
ax.axvline(train.index[-1], ls="--", lw=1, color=MUTED_RED)
ax.set_title("Séparation chronologique — frontière d'information")
ax.legend(); ax.grid(alpha=0.2); plt.show()
print("Train :", train.index.min().date(), "→", train.index.max().date(), len(train), "observations")
print("Test  :", test.index.min().date(), "→", test.index.max().date(), len(test), "observations")


## 3. Deux baselines raisonnables

### Baseline 1 — Persistance
$$\widehat y_{T+h}=y_T.$$

### Baseline 2 — Drift
La dérive moyenne est estimée **uniquement sur le train** :
$$\widehat\delta=\frac{y_T-y_1}{T-1},\qquad \widehat y_{T+h}=y_T+h\widehat\delta.$$

Une baseline mieux adaptée peut retirer une grande partie de l'avantage apparent d'un modèle sophistiqué.


In [ ]:
persistence = np.repeat(train.iloc[-1], h)
delta_hat = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
drift = train.iloc[-1] + np.arange(1, h + 1) * delta_hat

baseline_metrics = pd.DataFrame({
    "MAE": [mae(test, persistence), mae(test, drift)],
    "RMSE": [rmse(test, persistence), rmse(test, drift)],
}, index=["Persistance", "Drift"])
print(f"Dérive estimée sur train : {delta_hat:.6f} unité/période")
baseline_metrics


## 4. Modèles candidats : sélectionner sans regarder le test

Nous ajustons plusieurs ordres sur **train seulement**. AIC et BIC sont calculés sur exactement les mêmes données d'entraînement. La période test n'intervient pas encore.


In [ ]:
candidates = [(0,1,0), (1,1,0), (0,1,1), (1,1,1), (2,1,1), (1,1,2)]
rows = []
fits = {}
for order in candidates:
    fit = ARIMA(train, order=order, trend="t").fit()
    fits[order] = fit
    rows.append({"order": order, "AIC_train": fit.aic, "BIC_train": fit.bic})
selection = pd.DataFrame(rows).sort_values("AIC_train").reset_index(drop=True)
selection


### Décision avant ouverture du test
Le modèle principal du laboratoire est celui de plus petit AIC sur l'entraînement. Nous enregistrons aussi le choix BIC. Nous ne changeons plus cette décision après avoir vu le test.


In [ ]:
best_aic_order = tuple(selection.iloc[0]["order"])
best_bic_order = tuple(selection.sort_values("BIC_train").iloc[0]["order"])
print("Meilleur AIC :", best_aic_order)
print("Meilleur BIC :", best_bic_order)


## 5. Diagnostic résiduel : Ljung–Box correctement interprété

Ljung–Box teste conjointement
$$H_0:\rho_1=\rho_2=\cdots=\rho_m=0.$$

Pour les résidus d'un ARIMA($p,d,q$), Statsmodels permet d'indiquer `model_df=p+q`. Une grande p-value ne prouve ni l'indépendance ni la perfection du modèle : elle signifie seulement que le test ne détecte pas d'autocorrélation globale significative aux lags considérés.


In [ ]:
best = fits[best_aic_order]
resid = best.resid.dropna().iloc[5:]
model_df = best_aic_order[0] + best_aic_order[2]
ljung = acorr_ljungbox(resid, lags=[10, 20], model_df=model_df, return_df=True)
print("model_df = p + q =", model_df)
display(ljung)

fig, axs = plt.subplots(2, 1, figsize=(11, 7))
axs[0].plot(resid, color=ACADEMIC_BLUE)
axs[0].axhline(0, lw=0.8, color=NEUTRAL_GRAY)
axs[0].set_title("Résidus"); axs[0].grid(alpha=0.2)
a = acf(resid, nlags=20, fft=True)
bound = 1.96 / np.sqrt(len(resid))
markerline, stemlines, baseline = axs[1].stem(range(len(a)), a, basefmt=" ")
plt.setp(stemlines, color=PETROL); plt.setp(markerline, color=PETROL)
axs[1].axhline(bound, ls="--", color=WARM_GOLD, lw=0.9)
axs[1].axhline(-bound, ls="--", color=WARM_GOLD, lw=0.9)
axs[1].axhline(0, lw=0.7, color=NEUTRAL_GRAY)
axs[1].set_title("ACF des résidus"); axs[1].set_xlabel("Retard"); axs[1].grid(alpha=0.2)
plt.tight_layout(); plt.show()


## 6. Prévision multi-pas à origine fixe

À la date $T$, le modèle est entraîné **une seule fois**, puis il produit :
$$\widehat y_{T+1|T},\widehat y_{T+2|T},\ldots,\widehat y_{T+h|T}.$$

C'est différent du **walk-forward**, où une nouvelle observation devient disponible, le modèle peut être mis à jour, puis une nouvelle prévision de court horizon est produite. Le walk-forward sera étudié en profondeur dans **CH06**.


## 7. Ouvrir enfin le test : ARIMA versus persistance versus drift

Les trois méthodes sont comparées sur **exactement les mêmes 60 observations futures**. Nous ne cherchons pas à faire gagner ARIMA : si une baseline simple fait aussi bien ou mieux, c'est le résultat scientifique à conserver.


In [ ]:
pred = best.get_forecast(steps=h)
mean = pred.predicted_mean
ci = pred.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(train.index[-160:], train.iloc[-160:], label="Historique", color=NEUTRAL_GRAY)
ax.plot(test.index, test, label="Réalisé", color=ACADEMIC_BLUE)
ax.plot(mean.index, mean, label=f"ARIMA{best_aic_order}", color=PETROL)
ax.plot(test.index, drift, label="Drift", color=WARM_GOLD, ls="--")
ax.fill_between(mean.index, ci.iloc[:,0], ci.iloc[:,1], alpha=0.18, color=WARM_GOLD, label="Intervalle ARIMA 95 %")
ax.axvline(train.index[-1], ls="--", lw=0.8, color=MUTED_RED)
ax.legend(); ax.grid(alpha=0.2); ax.set_title("Fixed-origin multi-step forecast — même période test")
plt.show()

comparison = pd.DataFrame({
    "MAE": [mae(test, persistence), mae(test, drift), mae(test, mean)],
    "RMSE": [rmse(test, persistence), rmse(test, drift), rmse(test, mean)],
}, index=["Persistance", "Drift", f"ARIMA{best_aic_order}"])
comparison


### Leçon centrale
Un modèle peut sembler extrêmement performant face à une baseline trop faible, puis perdre presque tout avantage lorsqu'on utilise une baseline simple mais mieux adaptée au problème. Ici, la drift baseline et ARIMA sont presque ex æquo : **drift gagne légèrement en MAE, ARIMA légèrement en RMSE** sur cette fenêtre.


In [ ]:
mae_p = comparison.loc["Persistance", "MAE"]
mae_d = comparison.loc["Drift", "MAE"]
mae_a = comparison.loc[f"ARIMA{best_aic_order}", "MAE"]
print(f"Gain MAE ARIMA vs persistance : {100*(mae_p-mae_a)/mae_p:.2f} %")
print(f"Gain MAE ARIMA vs drift       : {100*(mae_d-mae_a)/mae_d:.3f} %")
print("Un gain de prévision n'est pas automatiquement un gain économique.")


## 8. AIC/BIC versus performance future : comparaison post-évaluation

Maintenant seulement que le protocole principal a été fixé et évalué, nous pouvons regarder pédagogiquement comment les candidats se classent sur le test. Cette table **ne sert pas à re-sélectionner rétroactivement le modèle**.


In [ ]:
eval_rows = []
for order, fit in fits.items():
    fc = fit.forecast(h)
    eval_rows.append({
        "order": order,
        "AIC_train": fit.aic,
        "BIC_train": fit.bic,
        "MAE_test": mae(test, fc),
        "RMSE_test": rmse(test, fc),
    })
posthoc = pd.DataFrame(eval_rows).sort_values("AIC_train").reset_index(drop=True)
posthoc


## 9. Intervalle de prévision : interpréter avec prudence

Un intervalle annoncé à 95 % est conditionnel au modèle estimé et à ses hypothèses. Il dépend de la spécification, de l'estimation, de la distribution des innovations et de la structure de variance. Si les résidus sont peu autocorrélés mais que leurs carrés restent structurés, la dynamique de moyenne ne suffit plus : **CH03 introduira ARCH/GARCH**.


## 10. Mini-laboratoire étudiant

1. Refaire la sélection avec le BIC. Le modèle change-t-il ?
2. Comparer AIC/BIC et `MAE_test` sans re-sélectionner le modèle sur le test.
3. Répéter l'expérience avec $h=20$ puis $h=80$. La conclusion sur persistance/drift/ARIMA est-elle stable ?
4. Ajouter ARIMA(2,1,2) aux candidats. Son AIC justifie-t-il la complexité supplémentaire ?
5. Expliquer pourquoi la baseline de persistance n'est pas toujours la meilleure baseline naïve.
6. Expliquer la différence entre fixed-origin multi-step et walk-forward.
7. Interpréter une grande p-value Ljung–Box sans écrire « les résidus sont indépendants ».
8. Expliquer pourquoi choisir le modèle qui minimise directement `MAE_test` puis annoncer ce même `MAE_test` comme performance finale serait méthodologiquement fragile.


In [ ]:
# Votre travail commence ici.
# Q1

# Q2

# Q3

# Q4

# Q5

# Q6

# Q7

# Q8 : réponse argumentée en Markdown.


## Checklist de maîtrise

Vous devez maintenant pouvoir expliquer :
- pourquoi la frontière train/test est une frontière d'information ;
- pourquoi AIC/BIC et erreur hors échantillon ne répondent pas à la même question ;
- pourquoi plusieurs baselines raisonnables sont nécessaires ;
- pourquoi une baseline drift peut être plus pertinente qu'une simple persistance ;
- ce que teste Ljung–Box et ce qu'une grande p-value **ne prouve pas** ;
- la différence entre forecast multi-pas à origine fixe et walk-forward ;
- pourquoi un intervalle de prévision est conditionnel au modèle ;
- pourquoi une amélioration statistique n'est pas encore une performance de trading.
